# Do molecular toxicity models learn context, or just a toxic-looking motif?

**Pilot experiment:** test whether a molecular model can distinguish toxic from non-toxic molecules *within the subset that shares a candidate structural motif*, evaluated on held-out Bemis–Murcko scaffolds.

This is a more direct test of context sensitivity than comparing random and scaffold splits alone.

## Hypothesis
If a model uses molecular context beyond a candidate motif, a model trained on molecular fingerprints should rank toxic above non-toxic molecules **among test molecules that all contain that motif**. A motif-presence-only model cannot distinguish examples inside this subset and therefore has AUC = 0.5 whenever both classes are present.

## Important interpretation
A good result supports context-sensitive prediction; it does **not by itself prove** that the model has learned causal chemistry or that the candidate motif is a shortcut. Toxicity labels are noisy and assay-specific, and other correlated features can still drive predictions. This is an exploratory pilot with predeclared candidate motifs and an explicit decision rule.

**Dataset:** Tox21 (12 assay endpoints), using its public SMILES and partially observed labels. The notebook tries to download the dataset; if network access is unavailable, set `DATA_PATH` to a local `tox21.csv` file and rerun from the data-loading cell.

**Model:** Morgan fingerprints + Random Forest.  
**Evaluation:** 5-fold GroupKFold by Bemis–Murcko scaffold; only out-of-fold predictions are scored.  
**Primary analysis:** ROC-AUC among held-out molecules containing each candidate motif, per assay.  
**Decision rule:** “PROMISING PILOT SIGNAL” only if there are at least 8 eligible motif–assay groups, the unweighted mean within-motif AUC is at least 0.60, and a scaffold-bootstrap 95% confidence interval for that mean has lower bound above 0.50. Otherwise report “NOT PROMISING” or “INCONCLUSIVE” according to data sufficiency. Thresholds are pragmatic pilot criteria, not universal scientific standards.

In [ ]:
# If needed, install dependencies in your notebook environment:
%pip install pandas numpy scikit-learn rdkit matplotlib requests

import os, io, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.utils import check_random_state

RANDOM_STATE = 42
DATA_PATH = None  # Optional: set to a local path, e.g. "tox21.csv"
TOX21_URLS = [
    "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz",
    "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/tox21.csv",
]

print("Dependencies imported. RDKit version:", __import__("rdkit").__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 54.0 MB/s eta 0:00:00
Dependencies imported. RDKit version: 2026.09.1


In [ ]:
# Load Tox21. If downloads fail, provide a local CSV through DATA_PATH.
data = None
load_errors = []

if DATA_PATH and os.path.exists(DATA_PATH):
    try:
        data = pd.read_csv(DATA_PATH)
        print(f"Loaded local file: {DATA_PATH} | shape={data.shape}")
    except Exception as e:
        load_errors.append(f"Local file error: {e}")

if data is None:
    for url in TOX21_URLS:
        try:
            print("Trying:", url)
            data = pd.read_csv(url)
            print(f"Loaded remote dataset | shape={data.shape}")
            break
        except Exception as e:
            load_errors.append(f"{url}: {type(e).__name__}: {e}")

if data is None:
    print("\nDATASET UNAVAILABLE in this runtime.")
    print("Download Tox21 separately and set DATA_PATH to the CSV, then rerun this cell.")
    for err in load_errors:
        print(" -", err)

if data is not None:
    print("Columns:", list(data.columns[:8]), "...", list(data.columns[-5:]))
    display(data.head())

Trying: https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz
Loaded remote dataset | shape=(7831, 14)
Columns: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE'] ... ['SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


## 1. Define candidate motifs before looking at results

These are deliberately varied examples, not claims that each is universally toxic. The SMARTS patterns are operational definitions and should be inspected/validated before interpreting results. Some motifs may be rare; the code will exclude motif–assay groups without enough examples.

The dataset is not modified to force an association between motif and toxicity. We ask a narrower question: among unseen-scaffold test molecules that share the motif, can the model distinguish assay-positive from assay-negative molecules?

In [ ]:
# Candidate motifs, defined in advance. SMARTS matching is binary: present at least once.
MOTIFS = {
    "nitro": "[N+](=O)[O-]",
    "aldehyde": "[CX3H1](=O)[#6]",
    "sulfonamide": "[SX4](=O)(=O)N",
    "aromatic_amine": "[c][NX3;H1,H2;!$(NC=O)]",
    "alkyl_halide": "[CX4][Cl,Br,I]",
    "epoxide_like": "[O;r3]1[C;r3][C;r3]1",
}
MOTIF_QUERIES = {name: Chem.MolFromSmarts(smarts) for name, smarts in MOTIFS.items()}
for name, query in MOTIF_QUERIES.items():
    if query is None:
        raise ValueError(f"Invalid SMARTS for {name}: {MOTIFS[name]}")
print("Motifs:", MOTIFS)

Motifs: {'nitro': '[N+](=O)[O-]', 'aldehyde': '[CX3H1](=O)[#6]', 'sulfonamide': '[SX4](=O)(=O)N', 'aromatic_amine': '[c][NX3;H1,H2;!$(NC=O)]', 'alkyl_halide': '[CX4][Cl,Br,I]', 'epoxide_like': '[O;r3]1[C;r3][C;r3]1'}


## 2. Prepare valid molecules, fingerprints, scaffolds, and motif indicators

We use Morgan fingerprints (radius 2, 2048 bits) as a reproducible molecular representation. Bemis–Murcko scaffolds are used as groups, so a scaffold's molecules stay in the same fold. Molecules that fail RDKit parsing are excluded and counted.

In [ ]:
if data is not None:
    if "smiles" not in data.columns:
        raise ValueError("Expected a 'smiles' column in Tox21 CSV.")
    label_cols = [c for c in data.columns if c != "smiles"]
    # Tox21 usually has 12 binary assay columns; ignore any non-assay metadata columns if present.
    label_cols = [c for c in label_cols if pd.api.types.is_numeric_dtype(data[c]) or
                  data[c].dropna().astype(str).isin(["0", "1", "0.0", "1.0"]).all()]
    # Keep columns with at least some binary observed labels.
    label_cols = [c for c in label_cols if data[c].dropna().isin([0, 1, 0.0, 1.0]).all() and data[c].notna().any()]

    valid_rows, mols, smiles_clean = [], [], []
    invalid_smiles = 0
    for idx, smi in enumerate(data["smiles"].astype(str)):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            invalid_smiles += 1
            continue
        valid_rows.append(idx)
        mols.append(mol)
        smiles_clean.append(Chem.MolToSmiles(mol))

    clean = data.iloc[valid_rows].copy().reset_index(drop=True)
    clean["canonical_smiles"] = smiles_clean
    clean = clean.drop_duplicates("canonical_smiles").reset_index(drop=True)
    # Rebuild molecules after duplicate removal for strict row alignment.
    mols = [Chem.MolFromSmiles(s) for s in clean["canonical_smiles"]]

    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    X = np.zeros((len(mols), 2048), dtype=np.uint8)
    for i, mol in enumerate(mols):
        arr = np.zeros((2048,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fpgen.GetFingerprint(mol), arr)
        X[i] = arr

    scaffolds = []
    for mol in mols:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffolds.append(Chem.MolToSmiles(scaffold) if scaffold is not None else "")
    groups = np.array(scaffolds)

    motif_df = pd.DataFrame(index=clean.index)
    for name, query in MOTIF_QUERIES.items():
        motif_df[name] = [int(mol.HasSubstructMatch(query)) for mol in mols]

    print(f"Rows in source: {len(data):,}")
    print(f"Invalid SMILES excluded: {invalid_smiles:,}")
    print(f"Unique valid molecules: {len(clean):,}")
    print(f"Unique Murcko scaffold groups: {len(np.unique(groups)):,}")
    print("Assays:", label_cols)
    display(motif_df.sum().sort_values(ascending=False).rename("molecules_with_motif").to_frame())

[17:27:29] WARNING: not removing hydrogen atom without neighbors
[17:27:29] Explicit valence for atom # 8 Al, 6, is greater than permitted
[17:27:30] Explicit valence for atom # 3 Al, 6, is greater than permitted
[17:27:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[17:27:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[17:27:30] Explicit valence for atom # 9 Al, 6, is greater than permitted
[17:27:30] Explicit valence for atom # 5 Al, 6, is greater than permitted
[17:27:31] Explicit valence for atom # 16 Al, 6, is greater than permitted
[17:27:31] Explicit valence for atom # 20 Al, 6, is greater than permitted
[17:27:32] WARNING: not removing hydrogen atom without neighbors


Rows in source: 7,831
Invalid SMILES excluded: 8
Unique valid molecules: 7,823
Unique Murcko scaffold groups: 2,404
Assays: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


,molecules_with_motif
aromatic_amine,748
alkyl_halide,372
nitro,343
sulfonamide,261
aldehyde,131
epoxide_like,70


## 3. Generate out-of-fold predictions on held-out scaffolds

For each assay, train only on rows with observed labels. Each molecule receives a prediction from a model that did not train on its scaffold. We use a fixed, modest Random Forest configuration to keep the pilot practical and reproducible.

**Primary subgroup:** test molecules that contain a given motif. We compute AUC only when that subgroup contains at least 10 positives and 10 negatives across out-of-fold predictions. We also report sample counts so small or imbalanced results are visible.

In [ ]:
if data is not None:
    MIN_CLASS_COUNT = 10
    N_SPLITS = 5

    oof = pd.DataFrame(index=clean.index)
    fold_ids = pd.Series(index=clean.index, dtype="float")
    assay_status = []

    for assay in label_cols:
        y_all = pd.to_numeric(clean[assay], errors="coerce")
        observed = y_all.notna() & y_all.isin([0, 1])
        idx = np.flatnonzero(observed.to_numpy())
        y = y_all.iloc[idx].astype(int).to_numpy()
        g = groups[idx]

        if len(np.unique(y)) < 2 or len(np.unique(g)) < N_SPLITS:
            assay_status.append({"assay": assay, "status": "skipped: insufficient classes/scaffolds", "n_observed": len(idx)})
            continue

        pred = np.full(len(idx), np.nan)
        local_fold = np.full(len(idx), -1)
        splitter = GroupKFold(n_splits=N_SPLITS)
        for fold, (tr, te) in enumerate(splitter.split(X[idx], y, groups=g)):
            if len(np.unique(y[tr])) < 2:
                continue
            clf = RandomForestClassifier(
                n_estimators=250,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE + fold,
            )
            clf.fit(X[idx[tr]], y[tr])
            pred[te] = clf.predict_proba(X[idx[te]])[:, 1]
            local_fold[te] = fold

        col = f"pred::{assay}"
        oof[col] = np.nan
        oof.loc[idx, col] = pred
        fold_ids.loc[idx] = local_fold
        assay_status.append({"assay": assay, "status": "ok" if np.isfinite(pred).sum() else "no predictions",
                             "n_observed": len(idx), "n_oof_predictions": int(np.isfinite(pred).sum()),
                             "overall_oof_auc": roc_auc_score(y[np.isfinite(pred)], pred[np.isfinite(pred)])
                             if np.isfinite(pred).sum() and len(np.unique(y[np.isfinite(pred)])) == 2 else np.nan})

    assay_status = pd.DataFrame(assay_status)
    display(assay_status)

,assay,status,n_observed,n_oof_predictions,overall_oof_auc
0,NR-AR,ok,7258,7258,0.784150
1,NR-AR-LBD,ok,6751,6751,0.869279
2,NR-AhR,ok,6542,6542,0.862732
3,NR-Aromatase,ok,5815,5815,0.764996
4,NR-ER,ok,6186,6186,0.715232
5,NR-ER-LBD,ok,6948,6948,0.777528
6,NR-PPAR-gamma,ok,6443,6443,0.774567
7,SR-ARE,ok,5825,5825,0.715085
8,SR-ATAD5,ok,7065,7065,0.786494
9,SR-HSE,ok,6460,6460,0.706047


## 4. Primary test: context sensitivity within motif-positive molecules

For each motif and assay, restrict to out-of-fold predictions for molecules that contain the motif. If the model predicts different outcomes within that group, it must be using information beyond the binary fact that the motif is present. The within-group AUC tests whether that extra information ranks observed assay outcomes correctly.

AUC = 0.5 is chance ranking. Importantly, this is evidence of predictive context use, not proof of causal chemistry: other correlated structural features may explain the result.

In [ ]:
results = []
if data is not None:
    for assay in label_cols:
        pcol = f"pred::{assay}"
        if pcol not in oof.columns:
            continue
        y = pd.to_numeric(clean[assay], errors="coerce")
        pred = oof[pcol]
        for motif in MOTIFS:
            mask = (motif_df[motif].astype(bool) & y.notna() & y.isin([0, 1]) & pred.notna())
            yy = y[mask].astype(int).to_numpy()
            pp = pred[mask].to_numpy()
            n_pos = int((yy == 1).sum())
            n_neg = int((yy == 0).sum())
            if n_pos >= MIN_CLASS_COUNT and n_neg >= MIN_CLASS_COUNT:
                auc = roc_auc_score(yy, pp)
                results.append({
                    "assay": assay, "motif": motif, "n": len(yy),
                    "n_positive": n_pos, "n_negative": n_neg,
                    "within_motif_auc": auc, "auc_minus_chance": auc - 0.5
                })

results = pd.DataFrame(results)
if len(results):
    display(results.sort_values("within_motif_auc", ascending=False).reset_index(drop=True))
    print(f"Eligible motif–assay groups: {len(results)}")
else:
    print("No eligible motif–assay groups. This may reflect rare motifs, small subgroups, or data-loading issues.")

,assay,motif,n,n_positive,n_negative,within_motif_auc,auc_minus_chance
0,NR-AR,alkyl_halide,345,12,333,0.909409,0.409409
1,NR-Aromatase,alkyl_halide,267,10,257,0.860700,0.360700
2,NR-AhR,alkyl_halide,319,15,304,0.830263,0.330263
3,SR-MMP,sulfonamide,180,23,157,0.801994,0.301994
4,SR-MMP,alkyl_halide,290,47,243,0.793976,0.293976
5,NR-AhR,aromatic_amine,575,193,382,0.793384,0.293384
6,NR-AR-LBD,alkyl_halide,320,13,307,0.789025,0.289025
7,NR-AhR,nitro,274,48,226,0.770004,0.270004
8,SR-p53,aromatic_amine,646,62,584,0.763809,0.263809
9,SR-MMP,aromatic_amine,544,126,418,0.747057,0.247057


Eligible motif–assay groups: 41


## Final decision cell: Is this research direction promising?

Run this cell **after the model and results cells**. It prints a plain-language decision based on the observed results, not on assumed or illustrative values.

The primary signal is whether the model can discriminate toxicity outcomes among held-out molecules that all contain a given motif. This supports context sensitivity, but does not alone prove shortcut learning.

**Predeclared pilot rule**
- At least 8 eligible motif–assay groups.
- Mean within-motif ROC-AUC ≥ 0.60.
- Approximate 95% bootstrap CI lower bound > 0.50.

These are exploratory screening thresholds, not universal standards.

In [ ]:
# FINAL AUTOMATIC VERDICT — run after the results cell.
# Uses real computed results; never treats missing data as a negative scientific result.

MIN_ELIGIBLE_GROUPS = 8
MIN_MEAN_AUC = 0.60
BOOTSTRAP_CI_LOWER_BOUND = 0.50

def bootstrap_mean_ci(values, n_boot=5000, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        boot_means[b] = rng.choice(values, size=len(values), replace=True).mean()
    return tuple(np.quantile(boot_means, [0.025, 0.975]))

print("=" * 68)
print("FINAL VERDICT: IS THE CONTEXT-SENSITIVITY IDEA PROMISING?")
print("=" * 68)

if data is None:
    verdict = "INCONCLUSIVE"
    explanation = (
        "Tox21 data was not loaded. No scientific conclusion can be drawn. "
        "Load the dataset and rerun the notebook."
    )
    print("Eligible motif–assay groups: unavailable")
    print("Mean within-motif AUC: unavailable")
    print("95% CI: unavailable")

elif "results" not in globals() or len(results) == 0:
    verdict = "INCONCLUSIVE"
    explanation = (
        "No motif–assay group met the minimum sample-size requirement. "
        "Try a local Tox21 CSV, inspect motif prevalence, or predefine more common motifs."
    )
    print("Eligible motif–assay groups: 0")
    print("Mean within-motif AUC: unavailable")
    print("95% CI: unavailable")

else:
    eligible_groups = len(results)
    aucs = results["within_motif_auc"].to_numpy(dtype=float)
    mean_auc = float(np.mean(aucs))
    ci_low, ci_high = bootstrap_mean_ci(aucs)

    print(f"Eligible motif–assay groups: {eligible_groups}")
    print(f"Mean within-motif ROC-AUC: {mean_auc:.3f}")
    print(f"Approx. group-bootstrap 95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
    print(f"Required groups: >= {MIN_ELIGIBLE_GROUPS}")
    print(f"Required mean AUC: >= {MIN_MEAN_AUC:.2f}")
    print(f"Required CI lower bound: > {BOOTSTRAP_CI_LOWER_BOUND:.2f}")

    if (eligible_groups >= MIN_ELIGIBLE_GROUPS
        and mean_auc >= MIN_MEAN_AUC
        and ci_low > BOOTSTRAP_CI_LOWER_BOUND):
        verdict = "PROMISING PILOT SIGNAL"
        explanation = (
            "The model contains predictive information beyond binary motif presence "
            "for the tested motif-positive groups on held-out scaffolds. This justifies "
            "a stronger follow-up study with a GNN, matched molecular pairs, and controlled "
            "correlation-shift tests."
        )
    elif eligible_groups < MIN_ELIGIBLE_GROUPS:
        verdict = "INCONCLUSIVE"
        explanation = (
            "There are too few eligible motif–assay groups to make a robust overall call. "
            "Do not interpret this as evidence against context sensitivity."
        )
    else:
        verdict = "NOT PROMISING UNDER THE PREDECLARED PILOT RULE"
        explanation = (
            "The required aggregate signal was not demonstrated. This does not prove "
            "that context is irrelevant; inspect per-motif counts, assay imbalance, and "
            "the full results before abandoning the direction."
        )

print("\\n" + "=" * 68)
print("RESULT:", verdict)
print("=" * 68)
print(explanation)

if data is not None and "results" in globals() and len(results):
    print("\\nBest-supported motifs by mean within-motif AUC (descriptive only):")
    summary = (results.groupby("motif")
               .agg(n_assays=("assay", "count"),
                    mean_auc=("within_motif_auc", "mean"),
                    total_examples=("n", "sum"))
               .sort_values("mean_auc", ascending=False))
    display(summary)

print(
    "\\nScientific caution: this is a pilot decision rule, not proof of shortcut learning. "
    "High within-motif AUC shows useful predictive information beyond motif presence; "
    "it may still come from correlated structural features. A controlled correlation-conflict "
    "experiment is needed to establish shortcut reliance more directly."
)

FINAL VERDICT: IS THE CONTEXT-SENSITIVITY IDEA PROMISING?
Eligible motif–assay groups: 41
Mean within-motif ROC-AUC: 0.663
Approx. group-bootstrap 95% CI: [0.627, 0.698]
Required groups: >= 8
Required mean AUC: >= 0.60
Required CI lower bound: > 0.50
\n====================================================================
RESULT: PROMISING PILOT SIGNAL
The model contains predictive information beyond binary motif presence for the tested motif-positive groups on held-out scaffolds. This justifies a stronger follow-up study with a GNN, matched molecular pairs, and controlled correlation-shift tests.
\nBest-supported motifs by mean within-motif AUC (descriptive only):


,n_assays,mean_auc,total_examples
motif,,,
alkyl_halide,12,0.717843,3733
aromatic_amine,12,0.686876,7264
nitro,9,0.633828,2537
epoxide_like,2,0.609974,116
sulfonamide,4,0.577998,826
aldehyde,2,0.553911,219


\nScientific caution: this is a pilot decision rule, not proof of shortcut learning. High within-motif AUC shows useful predictive information beyond motif presence; it may still come from correlated structural features. A controlled correlation-conflict experiment is needed to establish shortcut reliance more directly.


## 6. How to interpret the result

- **Promising:** the GNN/RF can distinguish assay outcomes among molecules that all share a candidate motif on unseen scaffolds. This supports the feasibility of a context-sensitivity study.
- **Not promising:** the selected model/motifs did not show robust within-motif discrimination under this protocol. This does not prove that context is irrelevant; the representation, motifs, labels, or sample sizes may be inadequate.
- **Inconclusive:** too few groups or no real data. Do not report this as a negative scientific finding.

### Important limitations before turning this into a paper
1. This notebook uses a Random Forest on Morgan fingerprints, not a GNN. Repeat with a GNN if the scientific target is graph neural networks.
2. Candidate motifs and SMARTS definitions need chemical review; a SMARTS match is not automatically a biologically meaningful toxicophore.
3. Within-motif AUC can reflect other correlated structural signals, not necessarily causal context.
4. Tox21 labels are assay-specific and partly missing. Never combine the 12 endpoints as if they were one toxicity label.
5. For stronger evidence, add chemically valid matched molecular pairs, a motif-only baseline, bootstrap confidence intervals at the molecule/scaffold level, and a held-out correlation-conflict experiment.
6. Do not cherry-pick the best motif after seeing results. If exploring all motifs, report the full table and correct for multiple comparisons.

**Bottom line:** this is a useful first falsification test for context sensitivity, not a standalone proof of shortcut learning.